# Visual QC + Evaluation (SAM3 Final)

Interactive notebook for browsing SAM3 outputs and (optionally) evaluating against GT.


In [1]:
# Paths (edit as needed)
IMAGE_DIR = "/media/data/building_instance_tamu/test/images"
OUTPUT_DIR = "/media/data/building_instance_tamu/sam3/test_260225"
GT_DIR = ""  # optional

# Optional: metadata table for georef recovery (csv/json)
METADATA_PATH = ""


In [2]:
import sys
from pathlib import Path
import json
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw
import rasterio
from rasterio.features import rasterize
import shapely.wkt
from shapely.geometry import shape
import geopandas as gpd
import ipywidgets as widgets
from IPython.display import display

sys.path.insert(0, str(Path(__file__).resolve().parents[1] / "src"))
from sam3_final.georef import find_georef
from sam3_final.viz import load_mask, colorize_instance_mask, draw_polygons, stitch_instance_masks, stitch_annotation_tiles


NameError: name '__file__' is not defined

In [ ]:
IMAGE_DIR = Path(IMAGE_DIR)
OUTPUT_DIR = Path(OUTPUT_DIR)
GT_DIR = Path(GT_DIR) if GT_DIR else None
METADATA_PATH = Path(METADATA_PATH) if METADATA_PATH else None

def list_image_ids():
    files = sorted(IMAGE_DIR.glob("*_pre_disaster.png"))
    if not files:
        files = sorted(IMAGE_DIR.glob("*.png"))
    return [p.stem for p in files]

def load_pred_features():
    geojson_path = OUTPUT_DIR / "buildings.geojson"
    gpkg_path = OUTPUT_DIR / "buildings.gpkg"
    if geojson_path.exists():
        with open(geojson_path) as f:
            data = json.load(f)
        rows = []
        for feat in data.get("features", []):
            geom = shape(feat["geometry"])
            props = feat.get("properties", {})
            rows.append({**props, "geometry": geom})
        return gpd.GeoDataFrame(rows, geometry="geometry")
    if gpkg_path.exists():
        return gpd.read_file(gpkg_path)
    return gpd.GeoDataFrame([], geometry="geometry")

def load_gt_features(image_id):
    if GT_DIR is None:
        return []

    # 1) WKT JSON format used in notebooks
    wkt_path = GT_DIR / f"{image_id}.json"
    if wkt_path.exists():
        with open(wkt_path) as f:
            data = json.load(f)
        feats = data.get("features", {}).get("xy", [])
        return [shapely.wkt.loads(f["wkt"]) for f in feats if "wkt" in f]

    # 2) GeoJSON
    geojson_path = GT_DIR / f"{image_id}.geojson"
    if geojson_path.exists():
        gdf = gpd.read_file(geojson_path)
        return list(gdf.geometry)

    # 3) GPKG / Shapefile
    gpkg_files = list(GT_DIR.glob("*.gpkg"))
    shp_files = list(GT_DIR.glob("*.shp"))
    for p in gpkg_files + shp_files:
        gdf = gpd.read_file(p)
        if "image_id" in gdf.columns:
            gdf = gdf[gdf["image_id"] == image_id]
        return list(gdf.geometry)

    return []

def load_mask_for_image(image_id, img_size):
    mask_path = OUTPUT_DIR / "masks" / f"{image_id}.tif"
    if mask_path.exists():
        return load_mask(mask_path)
    tile_masks = list((OUTPUT_DIR / "masks").glob(f"{image_id}_x*_y*_w*_h*.tif"))
    if not tile_masks:
        return None
    return stitch_instance_masks(tile_masks, img_size)


def load_annotation_for_image(image_id, img_size):
    ann_full = OUTPUT_DIR / "annotations" / f"{image_id}_ann_full.png"
    if ann_full.exists():
        return Image.open(ann_full)
    ann_path = OUTPUT_DIR / "annotations" / f"{image_id}_ann.png"
    if ann_path.exists():
        return Image.open(ann_path)
    tile_anns = list((OUTPUT_DIR / "annotations").glob(f"{image_id}_x*_y*_w*_h*_ann.png"))
    if not tile_anns:
        return None
    return stitch_annotation_tiles(tile_anns, img_size)


def get_pred_polys_for_image(gdf, image_id):
    if gdf.empty:
        return []
    if "image_id" in gdf.columns:
        g = gdf[gdf["image_id"] == image_id]
        return list(g.geometry)
    return list(gdf.geometry)

def rasterize_polygons(polys, shape_hw, transform=None):
    h, w = shape_hw
    if transform is None:
        # pixel-space draw using PIL
        im = Image.new("L", (w, h), 0)
        dr = ImageDraw.Draw(im)
        for poly in polys:
            if poly.is_empty:
                continue
            if poly.geom_type == "Polygon":
                coords = [(int(x), int(y)) for x, y in poly.exterior.coords]
                dr.polygon(coords, fill=1)
            elif poly.geom_type == "MultiPolygon":
                for g in poly.geoms:
                    coords = [(int(x), int(y)) for x, y in g.exterior.coords]
                    dr.polygon(coords, fill=1)
        return np.array(im)
    return rasterize([(g, 1) for g in polys], out_shape=(h, w), transform=transform, fill=0, dtype=np.uint8)

pred_gdf = load_pred_features()
image_ids = list_image_ids()
print(f"Found {len(image_ids)} images")


In [ ]:
def show_image(image_id):
    img_path = IMAGE_DIR / f"{image_id}.png"
    if not img_path.exists():
        print(f"Missing image: {img_path}")
        return
    img = Image.open(img_path)

    ann_img = load_annotation_for_image(image_id, img.size)

    mask = load_mask_for_image(image_id, img.size)
    mask_vis = colorize_instance_mask(mask) if mask is not None else None

    preds = get_pred_polys_for_image(pred_gdf, image_id)
    pred_overlay = draw_polygons(img, preds, color="yellow", width=2) if preds else img.copy()

    gt_polys = load_gt_features(image_id)
    if gt_polys:
        pred_overlay = draw_polygons(pred_overlay, gt_polys, color="white", width=2)

    fig, axes = plt.subplots(2, 2, figsize=(12, 10))
    axes = axes.ravel()

    axes[0].imshow(img); axes[0].set_title("Original"); axes[0].axis("off")
    if ann_img is not None:
        axes[1].imshow(ann_img); axes[1].set_title("SAM3 Annotation (stitched if tiled)"); axes[1].axis("off")
    else:
        axes[1].text(0.5, 0.5, "No annotation PNG", ha="center", va="center")
        axes[1].axis("off")

    if mask_vis is not None:
        axes[2].imshow(mask_vis); axes[2].set_title("Instance Mask"); axes[2].axis("off")
    else:
        axes[2].text(0.5, 0.5, "No mask", ha="center", va="center")
        axes[2].axis("off")

    axes[3].imshow(pred_overlay); axes[3].set_title("Pred (yellow) + GT (white)"); axes[3].axis("off")
    plt.tight_layout()
    plt.show()

def interactive_viewer():
    dd = widgets.Dropdown(options=image_ids, description="image_id")
    out = widgets.Output()
    def on_change(change):
        if change["name"] == "value":
            with out:
                out.clear_output(wait=True)
                show_image(change["new"])
    dd.observe(on_change)
    display(dd, out)
    if image_ids:
        dd.value = image_ids[0]

interactive_viewer()


In [ ]:
def compute_iou_for_image(image_id):
    img_path = IMAGE_DIR / f"{image_id}.png"
    if not img_path.exists():
        return None
    img = Image.open(img_path)
    w, h = img.size

    preds = get_pred_polys_for_image(pred_gdf, image_id)
    gt_polys = load_gt_features(image_id)
    if not gt_polys or not preds:
        return None

    georef = find_georef(img_path, metadata_path=METADATA_PATH)
    transform = georef.transform

    pred_mask = rasterize_polygons(preds, (h, w), transform=transform)
    gt_mask = rasterize_polygons(gt_polys, (h, w), transform=transform)

    inter = np.logical_and(pred_mask, gt_mask).sum()
    union = np.logical_or(pred_mask, gt_mask).sum()
    if union == 0:
        return 0.0
    return float(inter / union)

def write_qc_and_iou_summaries():
    qc_rows = []
    iou_rows = []

    for image_id in image_ids:
        preds = get_pred_polys_for_image(pred_gdf, image_id)
        num_polys = len(preds)
        num_instances = None
        mean_conf = None
        if not pred_gdf.empty and "confidence" in pred_gdf.columns and "image_id" in pred_gdf.columns:
            g = pred_gdf[pred_gdf["image_id"] == image_id]
            if len(g):
                mean_conf = float(g["confidence"].dropna().mean()) if g["confidence"].notna().any() else None
        mask = load_mask_for_image(image_id, Image.open(IMAGE_DIR / f"{image_id}.png").size)
        if mask is not None:
            num_instances = int(len(np.unique(mask)) - 1)
        qc_rows.append({
            "image_id": image_id,
            "num_instances": num_instances,
            "num_polygons": num_polys,
            "mean_confidence": mean_conf,
            "notes": ""
        })

        iou = compute_iou_for_image(image_id)
        if iou is not None:
            iou_rows.append({
                "image_id": image_id,
                "iou": iou,
            })

    qc_df = pd.DataFrame(qc_rows)
    qc_path = OUTPUT_DIR / "qc_summary.csv"
    qc_df.to_csv(qc_path, index=False)
    print(f"Wrote {qc_path}")

    if iou_rows:
        iou_df = pd.DataFrame(iou_rows)
        iou_path = OUTPUT_DIR / "iou_summary.csv"
        iou_df.to_csv(iou_path, index=False)
        print(f"Wrote {iou_path}")
    else:
        print("No IoU computed (missing GT or predictions).")

write_qc_and_iou_summaries()


In [ ]:
def export_montage(sample_n=20):
    out_dir = OUTPUT_DIR / "qc_report"
    out_dir.mkdir(parents=True, exist_ok=True)
    ids = random.sample(image_ids, min(sample_n, len(image_ids)))
    cols = 5
    rows = int(np.ceil(len(ids) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 3, rows * 3))
    axes = np.array(axes).ravel()
    for i, image_id in enumerate(ids):
        img_path = IMAGE_DIR / f"{image_id}.png"
        if img_path.exists():
            img = Image.open(img_path)
            preds = get_pred_polys_for_image(pred_gdf, image_id)
            overlay = draw_polygons(img, preds, color="yellow", width=2)
            axes[i].imshow(overlay)
            axes[i].set_title(image_id, fontsize=8)
        axes[i].axis("off")
    for j in range(i + 1, len(axes)):
        axes[j].axis("off")
    plt.tight_layout()
    out_path = out_dir / "qc_montage.png"
    fig.savefig(out_path, dpi=150)
    print(f"Wrote {out_path}")

# Example:
# export_montage(20)
